<a href="https://colab.research.google.com/github/Mogi21/pytorch-deep-learning-labs/blob/main/03_pytorch_wine_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import os
import zipfile

zip_path = 'Lab2_empty.zip'
if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extraction complete!")
else:
    print(f"Could not find {zip_path} in the workspace.")

Extracting Lab2_empty.zip...
Extraction complete!


In [18]:
import os
import torch as pt
import torch.nn as nn
import numpy as np
import pandas as pd
from collections import OrderedDict
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import f1_score

# ==========================================
# 1. DOWNLOAD & LOAD DATASET
# ==========================================
if not os.path.exists('winequality-red.csv'):
    !wget https://archive.ics.uci.edu/static/public/186/wine+quality.zip
    !unzip -o wine+quality.zip

wq_red = pd.read_csv('winequality-red.csv', sep=';')
wq_red['color'] = 'red'

wq_white = pd.read_csv('winequality-white.csv', sep=';')
wq_white['color'] = 'white'

wq = pd.concat([wq_red, wq_white]).sample(frac=1., random_state=42)
X = wq.drop('quality', axis=1)
y = wq['quality']

# ==========================================
# 2. PREPROCESSING & SPLITTING
# ==========================================
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=.2, stratify=y.values, shuffle=True, random_state=42
)
X_train_raw, X_valid_raw, y_train_raw, y_valid_raw = train_test_split(
    X_train_raw, y_train_raw, test_size=.1, stratify=y_train_raw.values, shuffle=True, random_state=42
)

train_colors = X_train_raw['color'].unique()
def encode_categorical(df_in: pd.DataFrame, df_out: pd.DataFrame) -> pd.DataFrame:
    for c in train_colors[:-1]:
        df_out[f'color_{c}'] = (df_in['color'].values == c).astype(float)
    return df_out

continuous_features = [c for c in X_train_raw.columns if c not in ['color', 'color_code']]
max_values = np.quantile(X_train_raw[continuous_features], .999, axis=0)

scaler = MinMaxScaler()
scaler.fit(np.minimum(X_train_raw[continuous_features], max_values))

def encode_continuous(df_in: pd.DataFrame, df_out: pd.DataFrame) -> pd.DataFrame:
    df_out[continuous_features] = scaler.transform(np.minimum(df_in[continuous_features], max_values))
    return df_out

def encode_data(df: pd.DataFrame) -> pd.DataFrame:
    output = pd.DataFrame()
    output = encode_categorical(df, output)
    output = encode_continuous(df, output)
    return output

X_train = encode_data(X_train_raw)
X_valid = encode_data(X_valid_raw)
X_test = encode_data(X_test_raw)

le = OneHotEncoder(sparse_output=False)
le.fit(y_train_raw.values[:, None])

def encode_labels(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(le.transform(df.values[:, None]), columns=le.categories_[0])

def decode_labels(y_proba: np.array) -> np.array:
    return np.array(le.categories_[0])[np.argmax(y_proba, axis=1)]

y_train = encode_labels(y_train_raw)
y_valid = encode_labels(y_valid_raw)
y_test = encode_labels(y_test_raw)

# ==========================================
# 3. DATALOADERS & DEVICE SETUP
# ==========================================
BATCH_SIZE = 32
device = pt.device("cuda" if pt.cuda.is_available() else "cpu")

loader_train = DataLoader([(_X, _y) for _X, _y in zip(X_train.values, y_train.values)], batch_size=BATCH_SIZE, shuffle=True)
loader_valid = DataLoader([(_X, _y) for _X, _y in zip(X_valid.values, y_valid.values)], batch_size=BATCH_SIZE, shuffle=False)
loader_test = DataLoader([(_X, _y) for _X, _y in zip(X_test.values, y_test.values)], batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. ARCHITECTURE (TASK 3)
# ==========================================
def create_network(dropout: float = 0.2) -> nn.Sequential:
    in_dim = X_train.values.shape[-1]
    out_dim = y_train.values.shape[-1]

    return nn.Sequential(OrderedDict([
        ('linear1', nn.Linear(in_features=in_dim, out_features=20)),
        ('relu1', nn.ReLU()),
        ('dropout1', nn.Dropout(p=dropout)),
        ('linear2', nn.Linear(in_features=20, out_features=15)),
        ('relu2', nn.ReLU()),
        ('dropout2', nn.Dropout(p=dropout)),
        ('linear3', nn.Linear(in_features=15, out_features=out_dim)),
        ('softmax', nn.Softmax(dim=-1))
    ]))

model = create_network(dropout=0.2).to(device)

# ==========================================
# 5. TRAINING INFRASTRUCTURE & EVALUATION
# ==========================================
def epoch(model, loader_train, optimizer, loss_fn):
    model.train()
    losses = []
    with pt.enable_grad():
        for X_batch, y_batch in loader_train:
            X_batch = X_batch.to(device, dtype=pt.float32)
            y_batch = y_batch.to(device, dtype=pt.float32)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            losses.append(loss.detach().cpu().numpy())
            loss.backward()
            optimizer.step()
    return np.mean(losses)

def evaluate(model, loader_valid, loss_fn=None):
    model.eval()
    labels, predictions, losses = [], [], []
    with pt.no_grad():
        for X_batch, y_batch in loader_valid:
            X_batch = X_batch.to(device, dtype=pt.float32)
            y_batch = y_batch.to(device, dtype=pt.float32)
            labels.extend(y_batch.cpu().numpy())
            y_pred = model(X_batch)
            predictions.extend(y_pred.cpu().numpy())
            if loss_fn is not None:
                losses.append(loss_fn(y_pred, y_batch).detach().cpu().numpy())

    f1 = f1_score(decode_labels(np.array(labels)), decode_labels(np.array(predictions)), average='macro')
    if loss_fn is None:
        return {'f1': f1}
    return {'loss': np.mean(losses), 'f1': f1}

# ==========================================
# 6. TRAINING WITH EARLY STOPPING (TASK 4)
# ==========================================
def fit(model, loader_train, loader_valid, epochs=100, lr=0.005, patience=5):
    optimizer = pt.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = pt.nn.CrossEntropyLoss()

    best_loss = float('inf')
    patience_counter = 0
    best_weights = None
    history = []

    for i in range(epochs):
        loss_train = epoch(model, loader_train, optimizer, loss_fn)
        metrics = evaluate(model, loader_valid, loss_fn)
        valid_loss = metrics['loss']

        history.append({
            'loss_train': loss_train,
            'loss_valid': valid_loss,
            'f1_valid': metrics['f1']
        })

        print(f'Epoch {i+1:d}/{epochs:d}:\tloss_train = {loss_train:.2f};\tloss_valid = {valid_loss:.2f};\tf1_valid = {metrics["f1"]:.2f};')

        if valid_loss < best_loss:
            best_loss = valid_loss
            patience_counter = 0
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n[!] Early stopping triggered at epoch {i+1}.")
                break

    if best_weights is not None:
        model.load_state_dict(best_weights)
        print("Restored best model parameters.")

    return pd.DataFrame(history)

print(f"Training on {device}...")
history = fit(model, loader_train, loader_valid, epochs=100, lr=0.005, patience=5)
print('Test F1:', evaluate(model, loader_test))

# ==========================================
# 7. GENERATE SUBMISSION FILE
# ==========================================
nextilearn_path = 'X_test.csv'
if os.path.exists(nextilearn_path):
    print(f"Loading custom evaluation set from {nextilearn_path}...")
    data_raw = pd.read_csv(nextilearn_path, index_col=0)
else:
    print(f"Warning: '{nextilearn_path}' not found. Falling back to the test dataset split (X_test_raw)....")
    data_raw = X_test_raw

data_nextilearn = [(_X, pt.nan) for _X in encode_data(data_raw).values]
loader_nextilearn = DataLoader(data_nextilearn, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
model.eval()

predictions = []
for X_batch, _ in loader_nextilearn:
    X_batch = X_batch.to(device, dtype=pt.float32)
    y_pred = decode_labels(model(X_batch).cpu().detach().numpy())
    predictions.extend(y_pred)

pd.DataFrame(predictions, columns=['predictions']).to_csv('submission.csv')
print("Predictions successfully saved to submission.csv!")

Training on cpu...
Epoch 1/100:	loss_train = 1.75;	loss_valid = 1.71;	f1_valid = 0.10;
Epoch 2/100:	loss_train = 1.71;	loss_valid = 1.69;	f1_valid = 0.13;
Epoch 3/100:	loss_train = 1.68;	loss_valid = 1.65;	f1_valid = 0.20;
Epoch 4/100:	loss_train = 1.66;	loss_valid = 1.64;	f1_valid = 0.20;
Epoch 5/100:	loss_train = 1.65;	loss_valid = 1.64;	f1_valid = 0.20;
Epoch 6/100:	loss_train = 1.64;	loss_valid = 1.64;	f1_valid = 0.20;
Epoch 7/100:	loss_train = 1.64;	loss_valid = 1.64;	f1_valid = 0.20;
Epoch 8/100:	loss_train = 1.64;	loss_valid = 1.65;	f1_valid = 0.19;
Epoch 9/100:	loss_train = 1.64;	loss_valid = 1.64;	f1_valid = 0.20;
Epoch 10/100:	loss_train = 1.63;	loss_valid = 1.64;	f1_valid = 0.20;

[!] Early stopping triggered at epoch 10.
Restored best model parameters.
Test F1: {'f1': 0.17508308579159435}
Loading custom evaluation set from X_test.csv...
Predictions successfully saved to submission.csv!
